# A short primer on spectral analysis

## time domain | frequency domain | periodogram | red noise | windows |

---

This optional side notebook explains, with small synthetic examples, what the power spectrum in the main practical does. There is no heavy maths: we build series whose answer we know, and check that the spectrum finds it.

**What you will learn**
* the *time domain* and the *frequency domain*, and period = 1/frequency;
* how to read a power spectrum (periodogram) plotted against period;
* why climate series are "red", and why a peak must stand out from red noise;
* what 81 years of data can and cannot tell you, and what a window does.

## 1. Time domain and frequency domain

So far we have looked at series as values over time: the **time domain**. Spectral analysis uses the **frequency domain**: it splits a series into a sum of sine waves of different periods and tells you how much of the variance each period carries. Think of an orchestra: which instruments are playing, and how loudly.

The **period** is the length of one cycle (years here); the **frequency** is cycles per year. **Period = 1 / frequency**, so a 5-year cycle has a frequency of 0.2 per year. Below, three sine waves (periods 5, 11 and 30 years) add up to an irregular-looking series.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import periodogram

plt.rcParams.update({"font.size": 12})
ORANGE, BLUE, GREY = "#D55E00", "#0072B2", "0.45"   # colour-blind-safe colours; grey for the null

# three sine waves and their sum
t = np.arange(0, 100, 0.1)                      # time in years (fine steps, just for a smooth plot)
waves = {5: 1.0, 11: 0.7, 30: 1.2}              # period (yr): amplitude
fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
for (period, amp), colour in zip(waves.items(), [ORANGE, BLUE, "0.3"]):
    axes[0].plot(t, amp * np.sin(2 * np.pi * t / period), color=colour, label=f"period {period} yr")
total = sum(amp * np.sin(2 * np.pi * t / period) for period, amp in waves.items())
axes[1].plot(t, total, color="k")
axes[0].set_title("Three sine waves")
axes[1].set_title("Their sum")
axes[0].legend(loc="upper right", ncol=3, fontsize=11)
for ax in axes:
    ax.set_ylabel("Value (arbitrary units)")
axes[1].set_xlabel("Time (years)")
plt.tight_layout()
plt.show()

## 2. A synthetic "climate" series and its power spectrum

We make **81 annual values** (the length of the series in the main practical): a 5-year cycle + a 20-year cycle + random noise. We plot it over time, then plot its power spectrum (`scipy.signal.periodogram`) against **period** on a log axis. Each point is one period that 81 values can resolve; tall points carry a lot of variance. Settings match the main notebook (Hann window, linear detrend; see Section 5). Dotted lines mark the cycles we put in.

**EDIT ME:** change the periods, amplitudes or `NOISE`. Raise `NOISE` to 2 or 3 and re-run: the peaks drown in the noise.

In [ ]:
# ---- EDIT ME ----
P1, A1 = 5, 1.0        # period (yr) and amplitude of the first cycle
P2, A2 = 20, 1.0       # period (yr) and amplitude of the second cycle
NOISE = 1.0            # standard deviation of the random noise
# -----------------

def spectrum(x, window="hann"):
    # periodogram of an annual series (1 value per year, linear detrend); returns period (yr) and power
    f, power = periodogram(x, fs=1.0, window=window, detrend="linear", axis=-1)
    return 1 / f[1:], power[..., 1:]            # drop frequency 0 (an infinite period)

def period_axis(ax):
    # log period axis with readable tick labels
    ax.set_xscale("log")
    ax.set_xticks([2, 3, 5, 10, 20, 40, 80]); ax.set_xticklabels([2, 3, 5, 10, 20, 40, 80]); ax.minorticks_off()
    ax.set_xlabel("Period (years)")

rng = np.random.default_rng(1)                   # fixed seed: same "random" noise every time
years = np.arange(81)
x = A1 * np.sin(2 * np.pi * years / P1) + A2 * np.sin(2 * np.pi * years / P2) + NOISE * rng.standard_normal(81)
period, power = spectrum(x)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(years, x, color=BLUE)
axes[0].set_xlabel("Time (years)"); axes[0].set_ylabel("Value (arbitrary units)")
axes[0].set_title("Time domain")
axes[1].plot(period, power, color=ORANGE, lw=2, marker="o", ms=3)
for p in (P1, P2):
    axes[1].axvline(p, color=GREY, ls=":")
    axes[1].annotate(f"{p}-yr cycle", (p, power.max()), xytext=(4, 0), textcoords="offset points", fontsize=11)
period_axis(axes[1])
axes[1].set_ylabel("Power (units² per cycle/yr)")
axes[1].set_title("Frequency domain (power spectrum)")
plt.tight_layout()
plt.show()

for lo, hi in [(3, 10), (10, 40)]:
    band = (period >= lo) & (period <= hi)
    print(f"Tallest point between {lo} and {hi} yr: period {period[band][np.argmax(power[band])]:.1f} yr")

## 3. White noise, red noise, and when a peak is real

**White noise** is independent random numbers: its spectrum is flat on average, with no preferred period. **Red noise** has memory: each year is partly a copy of the year before (an AR(1) process, here `PERSIST = 0.7`). The climate system has memory too (the ocean stores heat), so climate series are naturally red: most of their variance sits at **long periods**, even with no cycle at all. One spectrum of 81 values is very noisy, so the right panel averages 1000 spectra of each kind.

So a peak only counts if it **stands out from a red-noise background**. We simulate 1000 red-noise series with the same length, variance and lag-1 autocorrelation as our series and, at each period, take the value only 5% of them exceed: the dashed **95% red-noise line**.

Careful: the line is **pointwise** (a 5% chance at *each* period), so a series with no cycle can easily cross it somewhere by chance. The last printed line counts how often.

In [ ]:
PERSIST = 0.7                                     # memory of the red noise (0 = white noise)

def ar1(n, a, rng, n_series=None):
    # red noise: each value = a * previous value + fresh random noise (unit variance overall)
    shape = (n,) if n_series is None else (n_series, n)
    e = rng.standard_normal(shape)
    y = np.zeros(shape)
    y[..., 0] = e[..., 0]
    for i in range(1, n):
        y[..., i] = a * y[..., i - 1] + np.sqrt(1 - a**2) * e[..., i]
    return y

rng = np.random.default_rng(2)
white = rng.standard_normal(81)
red = ar1(81, PERSIST, rng)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(years, white, color=GREY, label="white noise")
axes[0].plot(years, red, color=ORANGE, label=f"red noise (persistence {PERSIST})")
axes[0].set_xlabel("Time (years)"); axes[0].set_ylabel("Value (standard units)")
axes[0].set_title("Time domain (one example of each)"); axes[0].set_ylim(-3, 3.8)
axes[0].legend(loc="upper left", ncol=2, fontsize=10)
# one spectrum of 81 values is very noisy, so we average the spectra of 1000 series of each kind
many_white = rng.standard_normal((1000, 81))
many_red = ar1(81, PERSIST, rng, n_series=1000)
for many, colour, name in [(many_white, GREY, "white noise"), (many_red, ORANGE, "red noise")]:
    p, pw = spectrum(many)
    axes[1].plot(p, pw.mean(axis=0), color=colour, lw=2, label=name)
period_axis(axes[1]); axes[1].set_yscale("log")
axes[1].set_ylabel("Mean power (units² per cycle/yr, log)")
axes[1].set_title("Frequency domain (average of 1000 spectra)"); axes[1].legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
N_SIM = 1000                                      # number of red-noise series for the background

def red_noise_95(x, seed=42):
    # pointwise 95th percentile of the spectra of N_SIM red-noise series that match x
    a = np.corrcoef(x[:-1], x[1:])[0, 1]          # lag-1 autocorrelation (the "memory") of x
    sims = ar1(len(x), a, np.random.default_rng(seed), n_series=N_SIM) * x.std()
    return a, np.percentile(spectrum(sims)[1], 95, axis=0)

a, null95 = red_noise_95(x)                       # x is the synthetic series from Section 2
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(period, power, color=ORANGE, lw=2, marker="o", ms=3, label="synthetic series")
ax.plot(period, null95, color=GREY, ls="--", lw=2, label="red noise, 95th percentile")
period_axis(ax)
ax.set_ylabel("Power (units² per cycle/yr)")
ax.set_title(f"Synthetic series vs red-noise background (lag-1 autocorr. {a:.2f})")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Periods (yr) where the series is above the line:", np.round(period[power > null95], 1))
# 1000 NEW pure red-noise series (no cycle at all): how many cross the line somewhere?
line = np.percentile(spectrum(ar1(81, PERSIST, np.random.default_rng(42), N_SIM))[1], 95, axis=0)
test = spectrum(ar1(81, PERSIST, np.random.default_rng(3), N_SIM))[1]
print(f"Pure red noise: {100 * np.mean((test > line).any(axis=1)):.0f}% of series cross the line at least once "
      f"(out of {len(period)} periods), by chance alone")

## 4. What 81 years can and cannot tell you

With 81 annual values the spectrum only contains the periods 81/1, 81/2, 81/3, ... years: frequencies are spaced by **1/length**. At long periods the points are far apart, and the linear detrend removes part of any very slow change, so cycles of about 40 years or longer **cannot be judged**. The cell below swaps the 20-year cycle for a **50-year** one: see where its power goes.

The spectrum measures *how much* variance each period carries, not sign or timing (phase): flipping the series upside down gives the same spectrum.

In [ ]:
rng = np.random.default_rng(1)
x50 = A1 * np.sin(2 * np.pi * years / P1) + A2 * np.sin(2 * np.pi * years / 50) + NOISE * rng.standard_normal(81)
p50, pow50 = spectrum(x50)
print("Longest periods the spectrum can show (yr):", np.round(np.sort(p50)[::-1][:5], 1))

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(p50, pow50, color=BLUE, lw=2, marker="o", ms=3)
ax.axvline(50, color=GREY, ls=":"); ax.annotate("50-yr cycle", (50, pow50.max()), xytext=(4, 0), textcoords="offset points", fontsize=11)
period_axis(ax)
ax.set_ylabel("Power (units² per cycle/yr)")
ax.set_title("A 50-year cycle in an 81-year series")
plt.tight_layout()
plt.show()

print("Same spectrum after flipping the sign of the series?", np.allclose(spectrum(x)[1], spectrum(-x)[1]))

## 5. A little more technical: windows and leakage

> The periodogram treats the series as if it repeated forever, so the jump between its last and first values acts like a sudden step that spreads power from a true peak into other periods (**leakage**). A **window** (taper) fades the series to zero at both ends, reducing this edge effect, at the price of a slightly wider, blurrier peak. Below, the two cycles of Section 2 without noise, so all power away from 5 and 20 years is leakage. The main notebook uses a Hann window with a linear detrend.

In [ ]:
clean = A1 * np.sin(2 * np.pi * years / P1) + A2 * np.sin(2 * np.pi * years / P2)   # the two cycles, no noise
fig, ax = plt.subplots(figsize=(8, 4.2))
for window, colour, name in [("boxcar", GREY, "boxcar (no taper)"), ("hann", ORANGE, "Hann window")]:
    ax.plot(*spectrum(clean, window=window), color=colour, lw=2, marker="o", ms=3, label=name)
period_axis(ax); ax.set_yscale("log")
ax.set_ylabel("Power (units² per cycle/yr, log scale)")
ax.set_title("The two cycles of Section 2 (no noise), two windows")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 6. Try it on your own series

`spectrum_plot(x)` takes any 1-D array of annual values and plots its spectrum with the 95% red-noise line. The example uses the red noise of Section 3, which has no cycle at all, yet the printout lists a few periods above the line. That is the chance crossing from Section 3 in action: a peak above the line is a hint, not a proof. Replace the example with your own array (e.g. a PC from the main notebook).

**Questions to think about**
1. How large must `NOISE` (Section 2) be before the 5-year peak no longer crosses the line?
2. Why would a single point above the line not convince you that a cycle is real?
3. An 81-year record has most power at its longest period. What can and can't you conclude?

In [ ]:
def spectrum_plot(x, title="My series"):
    # power spectrum of annual values x, with the pointwise 95% red-noise line
    x = np.asarray(x, dtype=float)
    period, power = spectrum(x)
    a, null95 = red_noise_95(x)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(period, power, color=BLUE, lw=2, marker="o", ms=3, label=title)
    ax.plot(period, null95, color=GREY, ls="--", lw=2, label="red noise, 95th percentile")
    period_axis(ax); ax.set_xlim(right=len(x))
    ax.set_ylabel("Power (units² per cycle/yr)")
    ax.set_title(f"{title} (lag-1 autocorr. {a:.2f})")
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
    print("Periods (yr) above the red-noise line:", np.round(period[power > null95], 1))

spectrum_plot(red, title="Red noise from Section 3")   # replace `red` with your own array